In [1]:
import numpy as np
import emcee
from scipy.stats import norm
import cosmo_kb_func as kbf
from multiprocessing import Pool
from pyswarms.single import GlobalBestPSO
#nohup jupyter nbconvert --to notebook --execute --inplace ./run_mcmc_bpl_fiducial.ipynb > run.log 2>&1 &

In [2]:
# --- Construct posterior probability model ---
def log_probability(theta, data, priors_config):
    """
     Compute the log-posterior probability for the given parameter vector theta.
    """
    # a. Unpack parameters
    a = theta[0] # parameters for gamma_ppn
    b, c, d = theta[1], theta[2], theta[3] # parameters for beta
    e, f = theta[4], theta[5] # parameters for delta_m

    lambda_val = priors_config['lambda_val'] #1.0 
    lb = priors_config['lower_bounds']
    ub = priors_config['upper_bounds']

    # b. Compute log prior ln(π(θ))      
    if not (lb[0] <= a <= ub[0]): return -np.inf
    log_prior_a = 0.0

    if not (lb[1] <= b <= ub[1] and lb[2] <= c <= ub[2] and lb[3] <= d <= ub[3]): return -np.inf
    log_prior_bc = 0.0

    if not (lb[4] <= e <= ub[4] and lb[5] <= f <= ub[5]): return -np.inf
    log_prior_de = 0.0

    log_prior_pi = log_prior_a + log_prior_bc + log_prior_de

    # c/d/e. Loop over all systems to compute likelihood and physical prior for each
    total_log_likelihood = 0.0
    total_log_physical_prior = 0.0
    
    num_systems = len(data['z_d']) 
    
    # use the following loop to choose systems for mcmcm fitting
    for j in range(num_systems):
        # c. Compute the model prediction for the j-th system
        weights_j = data['weights'][j]
        vbias_j = data['vbias'][j]
        delta_m = e + f*data['z_i'] #\delta_m(z)=d+e*z
        if np.any(delta_m <= -1): return -np.inf
        
        kappa_lss_j = np.sum(weights_j * delta_m) #external convergence from LSS
        mean_beta_j = b + c*data['z_d'][j] #\beta(z)=b+c*z
        if np.any(mean_beta_j >= 1): return -np.inf
        gamma_j = a 
        kext_rms_j = data['kext_rms'][j]
        kappa_env_j = data['kenv'][j] #external convergence from environment
        kappa_ext_j = kappa_lss_j + kappa_env_j
        xx = np.sqrt((1-kappa_ext_j)*2/(1+gamma_j))/vbias_j

        # d. Compute the log-likelihood for the j-th system
        interpolator = data['pdf_interpolators'][j]
        # point = np.array([beta_j, xx])
        # likelihood_val = interpolator(point)[0]
        beta_grid = interpolator.grid[0]
        pbeta_j = norm.pdf(beta_grid, loc=mean_beta_j, scale=d)
        points = np.column_stack((beta_grid, np.full_like(beta_grid,xx)))
        y = interpolator(points)*pbeta_j
        likelihood_val = np.trapezoid(y,beta_grid)        
        
        if likelihood_val <= 1e-20: return -np.inf
        total_log_likelihood += np.log(likelihood_val)
        
        # e. Compute physical prior for the j-th system
        total_log_physical_prior += norm.logpdf(kappa_lss_j, loc=0.0, scale=kext_rms_j)#log_p_theory_kappa(kappa_ext_j)

    # f. Return the total log-posterior probability
    total_log_prob = total_log_likelihood + lambda_val * total_log_physical_prior + log_prior_pi
    
    if not np.isfinite(total_log_prob): return -1e15 # Return a very large negative number instead of -inf

    return total_log_prob

In [3]:
# Parallel PSO---------------------------------
# --- 1. MCMC Settings and Data Acquisition  ---
arr = np.arange(125)
remove_list = [101,107,108,114] #4 outliers in the histogram of vel_model(beta=0)-vel_dr17 
filtered_arr = arr[~np.isin(arr, remove_list)]
num_systems=len(filtered_arr) 

#filtered_arr = None
if filtered_arr is None: num_systems = 101

MANIFEST_FILE_PATH = '../data/obs125_krms_vbias_kenv.dat'
FITS_DIRECTORY_PATH = '../data/xbeta_likelihood_maps/'
data = kbf.data_collection(MANIFEST_FILE_PATH,FITS_DIRECTORY_PATH,num_systems=num_systems, selected_indices=filtered_arr)

# --- 2. Objective Function and Boundary Settings ---

# pyswarms will minimize this function
def objective_function(theta, data, priors_config):
    return -log_probability(theta, data, priors_config)

# Wrapper function for pyswarms: takes the whole swarm
def pso_objective_wrapper(swarm_particles):
    #n_particles = swarm_particles.shape[0] #=n_particles/n_processes
    costs = [objective_function(p, data, priors_config) for p in swarm_particles]
    return np.array(costs)

# Define parameter bounds (gamma_ppn, beta_0, beta_z, tau_beta, delta_m,0, delta_m,z)
lower_bounds = np.array([0.5, -1.0, -5.0, 0.02, -1.0, -2.0])
upper_bounds = np.array([1.5, 1.0, 1.0, 0.5, 2.0, 2.0])

# pyswarms requires bounds as a tuple of (min_bounds, max_bounds)
bounds = (lower_bounds, upper_bounds)
lambda_val = 1.0#/num_systems

priors_config = {
    'lower_bounds': lower_bounds,
    'upper_bounds': upper_bounds,
    'lambda_val': lambda_val
}

# --- 3. Configure and Run Parallel PSO ---
print("--- Phase 1: Using pyswarms (parallel PSO) to find the best initial parameters... ---")

# PSO configuration
options = {'c1': 0.5, 'c2': 0.3, 'w': 0.9}

# Create a GlobalBestPSO optimizer
optimizer = GlobalBestPSO(
    n_particles=500,
    dimensions=len(lower_bounds),
    options=options,
    bounds=bounds
)

# Run the optimizer, specifying n_processes for parallel computation
best_score, best_params_pso = optimizer.optimize(
    pso_objective_wrapper, 
    iters=150,
    verbose = False,
    n_processes=100  # <--- Parallel computation is enabled here
)

print("PSO complete. The best parameter set found (MAP estimate):")
print(f"Best score (minimum negative log-probability): {best_score}")
print(f"Best parameters: {best_params_pso}")
print("-" * 50)

--- Loading data ---
Successfully loaded manifest file: ../data/obs125_krms_vbias_kenv.dat
Automatically ignored comment lines starting with '#', and loaded only the first 6 columns.
Found 125 lens systems.

Starting to load 125 FITS files...
Successfully loaded 125 / 125 PDFs.
--- Select data by index: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 102 103 104 105 106 109 110
 111 112 113 115 116 117 118 119 120 121 122 123 124] ---
Cosmology model in use: flat_lambda_CDM
Custom parameters: H0=67.4 km / (Mpc s), Om0=0.315
Total lookback time to z=1.52 is 9.581 Gyr
Total comoving distance to z=1.52 is 4519.469 Mpc

The number of 

In [4]:

# MCMC configuration
ndim = 6
nwalkers = 2000 
nsteps = 600

# Initialize the walkers' positions
initial_pos = best_params_pso + 1e-3 * np.random.randn(nwalkers, ndim)

# os.cpu_count() can be used to get the number of CPU cores on your machine
# Using a with statement ensures the process pool is properly closed after use
with Pool(80) as pool:
    print(f"MCMC will use {pool._processes} CPU cores for parallel computation...")
    
    # Create the emcee sampler and pass the pool object to it
    sampler = emcee.EnsembleSampler(
        nwalkers, 
        ndim, 
        log_probability, 
        args=[data, priors_config], 
        pool=pool  # <--- Create and pass the process pool
    )
    
    print(f"Starting MCMC sampling... N_dim={ndim}, N_walkers={nwalkers}, N_steps={nsteps}")
    
    # Run the MCMC
    # When a pool is present, emcee will automatically distribute tasks across all worker processes
    sampler.run_mcmc(initial_pos, nsteps, progress=False)

print("MCMC sampling completed.")

MCMC will use 80 CPU cores for parallel computation...
Starting MCMC sampling... N_dim=6, N_walkers=2000, N_steps=600


MCMC sampling completed.


In [5]:
samples=sampler.chain #nwalkers, nsteps, ndim
lnprobability=sampler.lnprobability #nwalkers, nsteps, ndim=1

file_name = "bpl_fiducial_chain"
np.savez(file_name, remove_list=remove_list, samples=samples, lnprob=lnprobability)
print("Data saved as " + file_name)

Data saved as bpl_fiducial_chain
